## S3 연결

In [1]:
# 라이브러리 로드
import boto3, os, io

In [2]:
# s3 연결
session = boto3.Session()
s3Client = session.client('s3')

In [3]:
bucket_name = 'hotel-pricing-s3-bucket'
folder_path = 'test/'

# 버킷 설정
s3 = boto3.resource('s3')
s3_bucket = s3.Bucket(bucket_name)

## 라이브러리 로드

In [4]:
import pandas as pd
import numpy as np
from pytimekr import pytimekr
from datetime import date, datetime, timedelta
import lightgbm as lgb

import joblib
import pickle
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 10)

import warnings
warnings.filterwarnings("ignore")

## 전일 시점의 예약률

In [5]:
yy = [datetime.today().strftime("%Y")]
mm = [datetime.today().strftime("%m"), (datetime.today()+timedelta(days=30)).strftime("%m"), (datetime.today()+timedelta(days=60)).strftime("%m")]

In [6]:
data = pd.DataFrame()

for i in yy:
      for j in mm:
            z = s3Client.get_object(Bucket = bucket_name, Key = folder_path + i + '_' + j+'.xls')
            table = pd.read_excel(io.BytesIO(z['Body'].read()), header=1)
            data = pd.concat([data,table])

In [7]:
hotel = ['제주','남원','경주','낙산','드림','도고','가평','블룸']
col = data.columns

In [8]:
new_col = ['Date','Day']
for i in hotel:
      for j in range(2, 9):
            ncol = i + col[j]
            new_col.append(ncol)

In [9]:
data.columns = new_col
data.index = data['Date']

In [10]:
now = datetime.today().strftime("%Y-%m-%d")
now = pd.to_datetime(now, format='%Y-%m-%d')

In [11]:
data_2 = data.copy()
data_2.Date = [str(date_) for date_ in data_2.Date]

## 예측 날짜 데이터 생성

In [12]:
start_date = now
end_date = now + timedelta(days=59)
date_list = pd.date_range(start_date, end_date)

In [13]:
price_plan1 = list(range(85, 106, 5)) #Occ 0~30%
price_plan2 = list(range(95, 116, 5)) #Occ 31~60%
price_plan3 = list(range(100, 121, 5)) #Occ 61~85%
price_plan4 = list(range(110, 131, 5)) #Occ 86~100%

In [14]:
y = [] 

for date_ in date_list : 
    date_ = date_.strftime("%Y-%m-%d")

    rate_ = data_2.query("Date == '{}'".format(date_))['제주Occ%'][0]

    if rate_ <= 30 : 
        y.append(price_plan1) 
    elif rate_ >30 and rate_ <= 60 : 
        y.append(price_plan2) 
    elif rate_ >60 and rate_ <= 85 : 
        y.append(price_plan3) 
    else : 
        y.append(price_plan4)

In [15]:
from math import ceil
def week_of_month(dt):
    first_day = dt.replace(day=1)
    dom = dt.day
    adjusted_dom = dom + first_day.weekday()
    return int(ceil(adjusted_dom/7.0))

data_frame = pd.DataFrame({"예약일":[], "리드타임":[], '숙박일':[]})


for l in range(0, 60):
    new_data = {"예약일" : now, "리드타임" : l, '숙박일' :  now +  timedelta(days = l)}
    data_frame = data_frame.append(new_data, ignore_index = True)
data_frame['리드타임'] = data_frame['리드타임'].astype('int')

def week_of_month(dt):
    first_day = dt.replace(day=1)
    dom = dt.day
    adjusted_dom = dom + first_day.weekday()
    return int(ceil(adjusted_dom/7.0))

# 예약일 구조화
data_frame['예약_year'] = data_frame['예약일'].apply(lambda x: x.year)
data_frame['예약_month'] = data_frame['예약일'].apply(lambda x: x.month)
data_frame['예약_day'] = data_frame['예약일'].apply(lambda x: x.day)
data_frame['예약_주차'] = data_frame['예약일'].apply(week_of_month)
data_frame['예약_요일'] = data_frame['예약일'].apply(lambda x: x.weekday())

#예약일 앞뒤 휴일 정보
data_frame['예약일-1d_red'] = data_frame['예약일'].apply(lambda x: x - timedelta(days = 1) ) 
data_frame['예약일+1d_red'] = data_frame['예약일'].apply(lambda x: x + timedelta(days = 1) ) 

previous = []
later = []
for i in range(0, len(data_frame)):
    previous.append(pytimekr.is_red_day(pd.to_datetime(data_frame['예약일-1d_red'][i])))
    later.append(pytimekr.is_red_day(pd.to_datetime(data_frame['예약일+1d_red'][i])))
    
data_frame['예약일-1d_red'] = previous
data_frame['예약일-1d_red'] = np.where(data_frame['예약일-1d_red'] == False, 0, 1)
data_frame['예약일+1d_red'] = later
data_frame['예약일+1d_red'] = np.where(data_frame['예약일+1d_red'] == False, 0, 1)
data_frame['예약_주말'] = np.where(data_frame['예약_요일'] == 4 | 5, 1, 0)

#숙박일 구조화
data_frame['숙박_year'] = data_frame['숙박일'].apply(lambda x: x.year)
data_frame['숙박_month'] = data_frame['숙박일'].apply(lambda x: x.month)
data_frame['숙박_day'] = data_frame['숙박일'].apply(lambda x: x.day)
data_frame['숙박_주차'] = data_frame['숙박일'].apply(week_of_month)
data_frame['숙박_요일'] = data_frame['숙박일'].apply(lambda x: x.weekday())

#숙박일 앞뒤 휴일 정보
data_frame['숙박일-1d_red'] = data_frame['숙박일'].apply(lambda x: x - timedelta(days = 1) ) 
data_frame['숙박일+1d_red'] = data_frame['숙박일'].apply(lambda x: x + timedelta(days = 1) ) 

previous = []
later = []
for i in range(0, len(data_frame)):
    previous.append(pytimekr.is_red_day(pd.to_datetime(data_frame['숙박일-1d_red'][i])))
    later.append(pytimekr.is_red_day(pd.to_datetime(data_frame['숙박일+1d_red'][i])))
    
data_frame['숙박일-1d_red'] = previous
data_frame['숙박일-1d_red'] = np.where(data_frame['숙박일-1d_red'] == False, 0, 1)
data_frame['숙박일+1d_red'] = later
data_frame['숙박일+1d_red'] = np.where(data_frame['숙박일+1d_red'] == False, 0, 1)
data_frame['숙박_주말'] = np.where(data_frame['숙박_요일'] == 4 | 5, 1, 0)

print(data_frame.shape)

(60, 19)


## 비즈니스 환경 변수 추가

### 예약 데이터

In [16]:
# 예약데이터 main
a = s3Client.get_object(Bucket = bucket_name, Key = folder_path + 'main_hotel_rsv.xlsx')
main_rsv_log = pd.read_excel(io.BytesIO(a['Body'].read()), header=0)
main_rsv_log = main_rsv_log.drop(['Unnamed: 0'], axis=1)

In [17]:
yesterday = datetime.today() - timedelta(1)
firstdate = yesterday.strftime("%Y%m%d")

# 예약데이터 add
b = s3Client.get_object(Bucket = bucket_name, Key = folder_path + 'GGL(Rsvn' + firstdate[-4:] + ').xls')
add_rsv_log = pd.read_excel(io.BytesIO(b['Body'].read()), header=0)
add_rsv_log = add_rsv_log.drop(['I/M/A'], axis=1)

In [18]:
# 마케팅 채널 GOT 
add_rsv_log = add_rsv_log[add_rsv_log['Ma Ty']!='EMP']
add_rsv_log = add_rsv_log[add_rsv_log['Ma Ty'].isin(['GOT'])]

# 논리적 오류 데이터 제거
add_rsv_log = add_rsv_log[(add_rsv_log['Nts'] != 0) & (add_rsv_log['Rms'] != 0)]
add_rsv_log.reset_index(inplace = True)

# 필요한 열만 선택
add_rsv_log = add_rsv_log[['Sts','Guest Name','Rsvn Date','Arr Date','Cancel Date','Nts','Rm Ty','Rms','Rm Rate','Account Name','Ma Ty','Ra Ty']]
add_rsv_log.columns = ['Sts','Guest Name','예약일','숙박일','취소일','Nts','Rm Ty','Rms','Rm Rate','Account Name','Ma Ty','Ra Ty']

In [19]:
# 객실유형 재분류
add_rsv_log['room_type'] = np.where(add_rsv_log['Rm Ty'].isin(['DDR','DTR']), 'DLX', np.where(~add_rsv_log['Rm Ty'].isin(['FBR','FTR','FDR']), 'etc', add_rsv_log['Rm Ty']))

# 1일 숙박료 평균 변수 추가
add_rsv_log['1day_mean_price'] = (add_rsv_log['Rm Rate'] / add_rsv_log['Nts']) /  add_rsv_log['Rms']

# 날짜 데이터 형 변환
add_rsv_log['예약일'] = add_rsv_log['예약일'].apply(lambda x: pd.to_datetime(str(x), format='%Y-%m-%d'))
add_rsv_log['숙박일'] = add_rsv_log['숙박일'].apply(lambda x: pd.to_datetime(str(x), format='%Y-%m-%d'))
add_rsv_log['취소일'] = add_rsv_log['취소일'].apply(lambda x: pd.to_datetime(str(x), format='%Y-%m-%d'))

#예약일 앞뒤 휴일 정보
add_rsv_log['예약일-1d_red'] = add_rsv_log['예약일'].apply(lambda x: x - timedelta(days = 1) ) 
add_rsv_log['예약일+1d_red'] = add_rsv_log['예약일'].apply(lambda x: x + timedelta(days = 1) ) 
previous = []
later = []
for i in range(0, len(add_rsv_log)):
    previous.append(pytimekr.is_red_day(pd.to_datetime(add_rsv_log['예약일-1d_red'][i])))
    later.append(pytimekr.is_red_day(pd.to_datetime(add_rsv_log['예약일+1d_red'][i])))
add_rsv_log['예약일-1d_red'] = previous
add_rsv_log['예약일+1d_red'] = later

#숙박일 앞뒤 휴일 정보
add_rsv_log['숙박일-1d_red'] = add_rsv_log['숙박일'].apply(lambda x: x - timedelta(days = 1) ) 
add_rsv_log['숙박일+1d_red'] = add_rsv_log['숙박일'].apply(lambda x: x + timedelta(days = 1) ) 
previous = []
later = []
for i in range(0, len(add_rsv_log)):
    previous.append(pytimekr.is_red_day(pd.to_datetime(add_rsv_log['숙박일-1d_red'][i])))
    later.append(pytimekr.is_red_day(pd.to_datetime(add_rsv_log['숙박일+1d_red'][i])))
add_rsv_log['숙박일-1d_red'] = previous
add_rsv_log['숙박일+1d_red'] = later

#숙박일 구조화
def week_of_month(dt):
    first_day = dt.replace(day=1)
    dom = dt.day
    adjusted_dom = dom + first_day.weekday()
    return int(ceil(adjusted_dom/7.0))

add_rsv_log['숙박_year'] = add_rsv_log['숙박일'].apply(lambda x: x.year)
add_rsv_log['숙박_month'] = add_rsv_log['숙박일'].apply(lambda x: x.month)
add_rsv_log['숙박_day'] = add_rsv_log['숙박일'].apply(lambda x: x.day)
add_rsv_log['숙박_주차'] = add_rsv_log['숙박일'].apply(week_of_month)
add_rsv_log['숙박_요일'] = add_rsv_log['숙박일'].apply(lambda x: x.weekday())

# 예약일 구조화
add_rsv_log['예약_year'] = add_rsv_log['예약일'].apply(lambda x: x.year)
add_rsv_log['예약_month'] = add_rsv_log['예약일'].apply(lambda x: x.month)
add_rsv_log['예약_day'] = add_rsv_log['예약일'].apply(lambda x: x.day)
add_rsv_log['예약_주차'] = add_rsv_log['예약일'].apply(week_of_month)
add_rsv_log['예약_요일'] = add_rsv_log['예약일'].apply(lambda x: x.weekday())

# 주말 (금,토)
add_rsv_log['숙박_주말'] = np.where(add_rsv_log['숙박_요일'] == 4 | 5, 1, 0)
add_rsv_log['예약_주말'] = np.where(add_rsv_log['예약_요일'] == 4 | 5, 1, 0)

# 리드타임 계산 및 생성
add_rsv_log['예약일_Lead_Time'] = (add_rsv_log['숙박일']- add_rsv_log['예약일']).dt.days
# 취소타임 계산 및 생성 v1
add_rsv_log['취소_Time'] = (add_rsv_log['숙박일']- add_rsv_log['취소일']).dt.days
add_rsv_log['취소_Time'] = add_rsv_log['취소_Time'].fillna(0)

#숙박 고객 수
add_rsv_log['숙박일_예약건수'] = 1

#예약 유지율
add_rsv_log['예약유지율'] = 1 - (add_rsv_log['취소_Time'] / add_rsv_log['예약일_Lead_Time'])
add_rsv_log['예약유지율'] = round(add_rsv_log['예약유지율'], 2)
add_rsv_log.tail(2)

,Sts,Guest Name,예약일,숙박일,취소일,Nts,Rm Ty,Rms,Rm Rate,Account Name,Ma Ty,Ra Ty,room_type,1day_mean_price,예약일-1d_red,...,숙박_month,숙박_day,숙박_주차,숙박_요일,예약_year,예약_month,예약_day,예약_주차,예약_요일,숙박_주말,예약_주말,예약일_Lead_Time,취소_Time,숙박일_예약건수,예약유지율
13,RC,JUNG HAEMI,2022-05-11,2022-05-31,2022-05-11,1,DDR,1,208800,부킹닷컴,GOT,FNB,DLX,208800.0,False,...,5,31,6,1,2022,5,11,3,2,0,0,20,20.0,1,0.00
14,RC,JUNG HAEMI,2022-05-11,2022-05-31,2022-05-12,1,DDR,1,208800,부킹닷컴,GOT,FNB,DLX,208800.0,False,...,5,31,6,1,2022,5,11,3,2,0,0,20,19.0,1,0.05


In [20]:
print('main_예약로그 shape',main_rsv_log.shape)
print('add_예약로그 shape',add_rsv_log.shape)
print('main_예약로 예약일 최소: ', main_rsv_log['예약일'].min())
print('main_예약로그 예약일 최대: ', main_rsv_log['예약일'].max())
print('add_예약로그 예약일 최소: ', add_rsv_log['예약일'].min())
print('add_예약로그 예약일 최대: ', add_rsv_log['예약일'].max())

main_예약로그 shape (21723, 34)
add_예약로그 shape (15, 34)
main_예약로 예약일 최소:  2020-01-01 00:00:00
main_예약로그 예약일 최대:  2022-05-11 00:00:00
add_예약로그 예약일 최소:  2022-05-11 00:00:00
add_예약로그 예약일 최대:  2022-05-11 00:00:00


In [21]:
full_rsv_log = pd.concat([main_rsv_log, add_rsv_log])
print('full_예약로그 예약일 최대: ', full_rsv_log['예약일'].min())
print('full_예약로그 예약일 최대: ', full_rsv_log['예약일'].max())

full_예약로그 예약일 최대:  2020-01-01 00:00:00
full_예약로그 예약일 최대:  2022-05-11 00:00:00


In [22]:
# full_rsv_log.to_excel('main_hotel_rsv.xlsx')
# s3_bucket.upload_file('main_hotel_rsv.xlsx', Key='test/main_hotel_rsv.xlsx')
# os.remove('main_hotel_rsv.xlsx')

### 이전날 예약 건수

In [23]:
got_record = pd.DataFrame(full_rsv_log[(full_rsv_log['room_type'] == 'DLX') & (full_rsv_log['Ma Ty'] == 'GOT')].groupby(['예약일'])['Sts'].count()).reset_index()
got_record.columns = ['예약일','예약일_전체거래']
got_record['예약일'] = got_record['예약일'].apply(lambda x: pd.to_datetime(str(x), format='%Y-%m-%d'))
data_frame["예약일_전체거래-1d"] = int(got_record[got_record['예약일'] == data_frame['예약일'][0] - timedelta(days = 1)]['예약일_전체거래'])
data_frame["예약일_전체거래-2d"] = int(got_record[got_record['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['예약일_전체거래'])
data_frame['예약량_변화량-1d'] = round((data_frame['예약일_전체거래-1d'] - data_frame['예약일_전체거래-2d']) / data_frame['예약일_전체거래-2d'], 2)
data_frame.tail(2)

,예약일,리드타임,숙박일,예약_year,예약_month,예약_day,예약_주차,예약_요일,예약일-1d_red,예약일+1d_red,예약_주말,숙박_year,숙박_month,숙박_day,숙박_주차,숙박_요일,숙박일-1d_red,숙박일+1d_red,숙박_주말,예약일_전체거래-1d,예약일_전체거래-2d,예약량_변화량-1d
58,2022-05-12,58,2022-07-09,2022,5,12,3,3,0,0,0,2022,7,9,2,5,0,1,1,30,16,0.88
59,2022-05-12,59,2022-07-10,2022,5,12,3,3,0,0,0,2022,7,10,2,6,1,0,0,30,16,0.88


### 이전달 60일 이내 예약 선호도

In [24]:
last_month_like = full_rsv_log[(full_rsv_log['숙박_year'] >= 2020) & (full_rsv_log['room_type'] == 'DLX') 
                               & (full_rsv_log['Ma Ty'] == 'GOT') & (full_rsv_log['예약일_Lead_Time'] <= 59)][['예약_year','예약_month', '숙박일_예약건수']]
last_month_like = pd.DataFrame(last_month_like.groupby(['예약_year','예약_month'])['숙박일_예약건수'].sum()).reset_index()
last_month_like.columns = ['예약_year','예약_month','60일내예약선호']
last_month_like = last_month_like[(last_month_like['예약_year'] == 2021) & (last_month_like['예약_month'] == 9)]
last_month_like.tail(5)
data_frame['이전달60일내예약선호'] = int(last_month_like['60일내예약선호'])

### 전년 동월 월별 60일 이내 리드타임에 따른 평균 예약유지율

In [25]:
# 지난 숙박일 만족도 평균
last_rsv_score = full_rsv_log[(full_rsv_log['예약_year'] == 2020) & (full_rsv_log['room_type'] == 'DLX') & (full_rsv_log['Ma Ty'] == 'GOT')][['예약_month','예약일_Lead_Time','예약유지율']]
last_rsv_score['예약유지율'] = last_rsv_score['예약유지율'].fillna(0)
last_rsv_score = last_rsv_score.groupby(['예약_month','예약일_Lead_Time'])['예약유지율'].mean().reset_index()
last_rsv_score.columns =['예약_month','리드타임','전년도_예약유지율']
last_rsv_score['전년도_예약유지율'] = round(last_rsv_score['전년도_예약유지율'], 2)
last_rsv_score = last_rsv_score[last_rsv_score['리드타임'] < 60]
last_rsv_score = last_rsv_score[last_rsv_score['예약_month'] == 10]
data_frame = pd.merge(data_frame, last_rsv_score, how = 'left', on = ['예약_month','리드타임'])
data_frame = data_frame.fillna(0)

## NAVER Trends

In [26]:
c = s3Client.get_object(Bucket='hotel-pricing-s3-bucket', Key = folder_path+'naver_trends_data.csv')
keyword = pd.read_csv(io.BytesIO(c['Body'].read()), header=0)
keyword_data = keyword.reindex(columns=['예약일','스위트_남','스위트_여','제주도호텔_남','제주도호텔_여','부킹닷컴_남','부킹닷컴_여',
                               '호텔스닷컴_남','호텔스닷컴_여','히든클리프_남','히든클리프_여','위호텔_남','위호텔_여'])
keyword_data['예약일'] = keyword_data['예약일'].apply(lambda x: pd.to_datetime(x, format='%Y-%m-%d'))

In [27]:
if keyword_data['예약일'].max() == pd.Timestamp(date.today() - timedelta(days=1)):
    data_frame["스위트_남-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 1)]['스위트_남'])
    data_frame["스위트_남-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['스위트_남'])
    data_frame["스위트_여-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 1)]['스위트_여'])
    data_frame["스위트_여-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['스위트_여'])

    data_frame["제주도호텔_남-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 1)]['제주도호텔_남'])
    data_frame["제주도호텔_남-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['제주도호텔_남'])
    data_frame["제주도호텔_여-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 1)]['제주도호텔_여'])
    data_frame["제주도호텔_여-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['제주도호텔_여'])

    data_frame["부킹닷컴_남-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 1)]['부킹닷컴_남'])
    data_frame["부킹닷컴_남-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['부킹닷컴_남'])
    data_frame["부킹닷컴_여-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 1)]['부킹닷컴_여'])
    data_frame["부킹닷컴_여-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['부킹닷컴_여'])

    data_frame["호텔스닷컴_남-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 1)]['호텔스닷컴_남'])
    data_frame["호텔스닷컴_남-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['호텔스닷컴_남'])
    data_frame["호텔스닷컴_여-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 1)]['호텔스닷컴_여'])
    data_frame["호텔스닷컴_여-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['호텔스닷컴_여'])

    data_frame["히든클리프_남-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 1)]['히든클리프_남'])
    data_frame["히든클리프_남-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['히든클리프_남'])
    data_frame["히든클리프_여-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 1)]['히든클리프_여'])
    data_frame["히든클리프_여-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['히든클리프_여'])

    data_frame["위호텔_남-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 1)]['위호텔_남'])
    data_frame["위호텔_남-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['위호텔_남'])
    data_frame["위호텔_여-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 1)]['위호텔_여'])
    data_frame["위호텔_여-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['위호텔_여'])

else:
    data_frame["스위트_남-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['스위트_남'])
    data_frame["스위트_남-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 3)]['스위트_남'])
    data_frame["스위트_여-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['스위트_여'])
    data_frame["스위트_여-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 3)]['스위트_여'])

    data_frame["제주도호텔_남-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['제주도호텔_남'])
    data_frame["제주도호텔_남-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 3)]['제주도호텔_남'])
    data_frame["제주도호텔_여-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['제주도호텔_여'])
    data_frame["제주도호텔_여-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 3)]['제주도호텔_여'])

    data_frame["부킹닷컴_남-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['부킹닷컴_남'])
    data_frame["부킹닷컴_남-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 3)]['부킹닷컴_남'])
    data_frame["부킹닷컴_여-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['부킹닷컴_여'])
    data_frame["부킹닷컴_여-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['부킹닷컴_여'])

    data_frame["호텔스닷컴_남-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['호텔스닷컴_남'])
    data_frame["호텔스닷컴_남-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 3)]['호텔스닷컴_남'])
    data_frame["호텔스닷컴_여-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['호텔스닷컴_여'])
    data_frame["호텔스닷컴_여-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 3)]['호텔스닷컴_여'])

    data_frame["히든클리프_남-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['히든클리프_남'])
    data_frame["히든클리프_남-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 3)]['히든클리프_남'])
    data_frame["히든클리프_여-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['히든클리프_여'])
    data_frame["히든클리프_여-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 3)]['히든클리프_여'])

    data_frame["위호텔_남-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['위호텔_남'])
    data_frame["위호텔_남-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 3)]['위호텔_남'])
    data_frame["위호텔_여-1d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 2)]['위호텔_여'])
    data_frame["위호텔_여-2d"] = float(keyword_data[keyword_data['예약일'] == data_frame['예약일'][0] - timedelta(days = 3)]['위호텔_여'])

## 홈페이지 방문자 수

In [28]:
d = s3Client.get_object(Bucket = bucket_name, Key = folder_path+'homepage_'+(datetime.today()-timedelta(days = 1)).strftime("%Y_%m")  + '.xlsx')
homepage_visit = pd.read_excel(io.BytesIO(d['Body'].read()), header=0)
homepage_visit = homepage_visit.rename(columns={'날짜':'예약일'})
homepage_visit['예약일'] = pd.to_datetime(homepage_visit['예약일'])

homepage_visit['예약일'] = homepage_visit['예약일'] + timedelta(days = 1)

In [29]:
data_frame1 = data_frame.join(homepage_visit.set_index('예약일')[['방문자수','방문횟수','신규방문자수','재방문자수','방문당PV']], on='예약일')
data_frame2 = data_frame1.rename(columns={'방문자수':'방문자수-1d', '방문횟수':'방문횟수-1d', '신규방문자수':'신규방문자수-1d', '재방문자수':'재방문자수-1d', '방문당PV':'방문당PV-1d'})

## 경쟁호텔 객실료

In [30]:
rpa_file = 'RPA_' + datetime.today().strftime("%y%m%d")+'.xlsx'

f = s3Client.get_object(Bucket = bucket_name, Key = folder_path + rpa_file)
add_price = pd.read_excel(io.BytesIO(f['Body'].read()), header=0)

In [31]:
add_price = add_price[['예약일','입실연도','입실월','입실일','퇴실일','제주 교원 스위트호텔','제주 신라 호텔','롯데 호텔 제주','히든크리프 호텔 네이쳐','위 호텔 제주']]
add_price = add_price.fillna(0)
add_price['제주 교원 스위트호텔'] = np.where(add_price['제주 교원 스위트호텔'] == '가격 정보 없음', 0, add_price['제주 교원 스위트호텔'])
add_price['제주 신라 호텔'] = np.where(add_price['제주 신라 호텔'] == '가격 정보 없음', 0, add_price['제주 신라 호텔'])
add_price['롯데 호텔 제주'] = np.where(add_price['롯데 호텔 제주'] == '가격 정보 없음', 0, add_price['롯데 호텔 제주'])
add_price['히든크리프 호텔 네이쳐'] = np.where(add_price['히든크리프 호텔 네이쳐'] == '가격 정보 없음', 0, add_price['히든크리프 호텔 네이쳐'])
add_price['위 호텔 제주'] = np.where(add_price['위 호텔 제주'] == '가격 정보 없음', 0, add_price['위 호텔 제주'])
add_price.columns = ['예약일','입실연도','입실월','입실일','퇴실일','스위트H','신라H','롯데H','히든크리프H','위H']

for i in range(len(add_price)):
    if add_price['히든크리프H'][i] == 0 :
        add_price['히든크리프H'][i] = add_price['히든크리프H'][i+1]
    elif add_price['히든크리프H'][i] == '가격 정보 없음' :
        add_price['히든크리프H'][i] = add_price['히든크리프H'][i-1]
add_price['히든크리프H'] = add_price['히든크리프H'].astype('int')

for i in range(len(add_price)):
    if add_price['히든크리프H'][i] >= 500000 :
        add_price['히든크리프H'][i] = add_price['히든크리프H'][i+1]
    elif add_price['히든크리프H'][i] >= 500000 :
        add_price['히든크리프H'][i] = add_price['히든크리프H'][i-1]

for i in range(len(add_price)):
    if add_price['스위트H'][i] == 0 :
        add_price['스위트H'][i] = add_price['스위트H'][i+1]
    elif add_price['스위트H'][i] == 0 :
        add_price['스위트H'][i] = add_price['스위트H'][i-1]
add_price['스위트H'] = add_price['스위트H'].astype('int')

for i in range(len(add_price)):
    if add_price['신라H'][i] == 0 :
        add_price['신라H'][i] = add_price['신라H'][i+1]
    elif add_price['신라H'][i] == 0 :
        add_price['신라H'][i] = add_price['신라H'][i-1]
add_price['신라H'] = add_price['신라H'].astype('int')

for i in range(len(add_price)):
    if add_price['롯데H'][i] == 0 :
        add_price['롯데H'][i] = add_price['롯데H'][i+1]
    elif add_price['롯데H'][i] == 0 :
        add_price['롯데H'][i] = add_price['롯데H'][i-1]
add_price['롯데H'] = add_price['롯데H'].astype('int')

for i in range(len(add_price)):
    if add_price['위H'][i] == 0 :
        add_price['위H'][i] = add_price['위H'][i+1]
    elif add_price['위H'][i] == 0 :
        add_price['위H'][i] = add_price['위H'][i-1]
add_price['위H'] = add_price['위H'].astype('int')

for i in range(len(add_price)):
    if add_price['위H'][i] >= add_price['히든크리프H'][i] :
        add_price['위H'][i] = add_price['히든크리프H'][i]   

add_price['예약일'] = add_price['예약일'].apply(lambda x: pd.to_datetime(x, format='%Y-%m-%d'))
add_price['숙박일'] = add_price['입실연도'].astype('str') + '-' + add_price['입실월'].astype('str') + '-' + add_price['입실일'].astype('str')
add_price['숙박일'] = add_price['숙박일'].apply(lambda x: pd.to_datetime(x, format='%Y-%m-%d'))
add_price['리드타임'] = (add_price['숙박일']- add_price['예약일']).dt.days
add_price = add_price.drop(['숙박일','입실연도','입실월','입실일','퇴실일','스위트H'],axis = 1)
add_price['경쟁사가격차이'] = add_price['히든크리프H'] - add_price['위H']

In [32]:
full_price = add_price.copy()

#full_price = pd.concat([main_price, add_price])
print('full_price 예약일 최소',full_price['예약일'].min())
print('full_price 예약일 최대',full_price['예약일'].max())

full_price 예약일 최소 2022-05-12 00:00:00
full_price 예약일 최대 2022-05-12 00:00:00


In [33]:
data_frame = pd.merge(data_frame2, full_price, how = 'left', on = ['예약일','리드타임'])

## 예측

In [34]:
data_frame['기준가격'] = (data_frame['히든크리프H']*0.6 + data_frame['위H']*0.4)

In [35]:
for i in range(len(data_frame)):
    if data_frame['경쟁사가격차이'][i] >= 200000 :
        data_frame['기준가격'][i] = (data_frame['히든크리프H'][i+1]*0.6) + (data_frame['위H'][i+1]*0.4)

In [36]:
df_2 = data_frame

In [37]:
full_df = pd.DataFrame()

for date_, price_ in zip(data_frame['숙박일'], y ): 
    mini_df = data_frame.query("숙박일 == '{}'".format(date_)) 

    for multiple in range(3) : 
        mini_df = pd.concat([mini_df, mini_df]) 
    mini_df = mini_df.head(5)
    
    mini_df['제안가격'] = list(pd.Series(price_)/100)
    
    full_df = pd.concat([full_df, mini_df]) 

In [38]:
full_df = full_df.reset_index(drop=True)

In [39]:
pred_data = full_df

In [40]:
pred_day = ['예약일', '숙박일','히든크리프H','위H','경쟁사가격차이','기준가격','제안가격']
pred_condition = ['리드타임','예약_주차','예약_요일','예약일-1d_red','예약일+1d_red','예약_주말', '숙박_주차','숙박_요일','숙박일-1d_red','숙박일+1d_red','숙박_주말','제안가격']
pred_value = ['예약일_전체거래-1d', '예약일_전체거래-2d', '예약량_변화량-1d', '이전달60일내예약선호', '전년도_예약유지율',
       '스위트_남-1d', '스위트_남-2d', '스위트_여-1d', '스위트_여-2d', '제주도호텔_남-1d',
       '제주도호텔_남-2d', '제주도호텔_여-1d', '제주도호텔_여-2d', '부킹닷컴_남-1d', '부킹닷컴_남-2d',
       '부킹닷컴_여-1d', '부킹닷컴_여-2d', '호텔스닷컴_남-1d', '호텔스닷컴_남-2d', '호텔스닷컴_여-1d',
       '호텔스닷컴_여-2d', '히든클리프_남-1d', '히든클리프_남-2d', '히든클리프_여-1d', '히든클리프_여-2d',
       '위호텔_남-1d', '위호텔_남-2d', '위호텔_여-1d', '위호텔_여-2d', '방문자수-1d', '방문횟수-1d',
       '신규방문자수-1d', '재방문자수-1d', '방문당PV-1d','신라H','롯데H', '히든크리프H', '위H', '경쟁사가격차이']

In [41]:
pred_day_data = pred_data[pred_day]
pred_condition_data = pred_data[pred_condition]
pred_value_data = pred_data[pred_value]

In [42]:
print(pred_condition_data.shape)
print(pred_value_data.shape)

(300, 12)
(300, 39)


### 예약율 모델 로드

In [43]:
g1 = s3Client.get_object(Bucket = bucket_name, Key = folder_path + '20211219_cl_scaler.pkl')
cl_scaler = joblib.load(io.BytesIO(g1['Body'].read()))

g2 = s3Client.get_object(Bucket = bucket_name, Key = folder_path + '20211219_cl_model.pkl')
cl_model = joblib.load(io.BytesIO(g2['Body'].read()))

### 예약유지율 모델 로드

In [44]:
h1 = s3Client.get_object(Bucket = bucket_name, Key = folder_path + '20211219_re_scaler.pkl')
re_scaler = joblib.load(io.BytesIO(h1['Body'].read()))

h2 = s3Client.get_object(Bucket = bucket_name, Key = folder_path + '20211219_re_model.pkl')
re_model = joblib.load(io.BytesIO(h2['Body'].read()))

## 예약율, 예약유지율 예측

In [45]:
cl_pred_dat_value_scale = cl_scaler.transform(pred_value_data)
cl_pred_value_data = pd.DataFrame(cl_pred_dat_value_scale, columns = pred_value_data.columns)

cl_pred_data_total = pd.concat([pred_condition_data, cl_pred_value_data], axis = 1)

re_pred_dat_value_scale = re_scaler.transform(pred_value_data)
re_pred_value_data = pd.DataFrame(re_pred_dat_value_scale, columns = pred_value_data.columns)

re_pred_data_total = pd.concat([pred_condition_data, re_pred_value_data], axis = 1)

In [46]:
rsv = pd.DataFrame(cl_model.predict_proba(cl_pred_data_total)[:,1], columns=['예약율'])
rsv = round(rsv, 3)

maintain = pd.DataFrame(re_model.predict(re_pred_data_total), columns=['예약유지율'])
maintain = round(maintain, 3)

In [48]:
price = result.drop(columns=['경쟁사가격차이'])
price.rename(columns = {'히든크리프H':'히든', '위H':'WE', '기준가격':'기준객실료', '제안가격':'상대비율', '예약율':'컨버전율', '산출_제안가격':'제안객실료'}, inplace = True)
price['예약일'] = price['예약일'].dt.date
price['숙박일'] = price['숙박일'].dt.date
price['기준객실료'] = round(price['기준객실료']).astype(int)
price['제안객실료'] = price['제안객실료'].astype(int)
price['점수'] = round(price['점수']).astype(int)

In [49]:
price.to_excel('download/price.xlsx', index=False)
s3_bucket.upload_file('download/price.xlsx', Key= folder_path +'price_'+now.strftime("%Y%m%d")+'.xlsx')
os.remove('download/price.xlsx')